# FairDerm: DDI Dataset Evaluation (Multi-Seed)

This notebook evaluates trained models on the **Diverse Dermatology Images (DDI)** dataset.

**Purpose:** Run inference on DDI external dataset for **both seeds (42 and 123)** and compute aggregated results (mean ± std).

**Models evaluated:**
- Baseline
- Mixup
- Reweighted
- Focal Loss
- Proposed

**DDI Dataset:**
- 656 images with verified pathology labels
- Near-balanced skin tone distribution (Light: 208, Medium: 241, Dark: 207)
- Binary classification: benign vs malignant

## 1. Setup

In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Install dependencies
!pip install timm --quiet

In [3]:
import sys
sys.path.insert(0, '/content/drive/MyDrive/thesis/code')

import os
import json
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader
from tqdm import tqdm
from pathlib import Path

# Imports from custom fairderm package
from fairderm import (
    SkinLesionClassifier,
    DDIDataset,
    compute_metrics,
    compute_fairness_metrics,
    print_fairness_report,
    get_ddi_eval_transforms,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


## 2. Configuration

In [4]:
# Paths
DDI_IMAGES_DIR = '/content/drive/MyDrive/thesis/data/ddidiversedermatologyimages'
DDI_METADATA_PATH = '/content/drive/MyDrive/thesis/data/ddi_metadata.csv'
RESULTS_DIR = Path('/content/drive/MyDrive/thesis/results')
DDI_RESULTS_DIR = RESULTS_DIR / 'ddi_evaluation'

os.makedirs(DDI_RESULTS_DIR, exist_ok=True)

# Multi-seed configuration
SEEDS = [42, 123]
MODELS = ['baseline', 'mixup', 'reweighted', 'focalloss', 'proposed']

print(f"DDI Images: {DDI_IMAGES_DIR}")
print(f"Models directory: {RESULTS_DIR}")
print(f"Results will be saved to: {DDI_RESULTS_DIR}")
print(f"\nSeeds to evaluate: {SEEDS}")
print(f"Models to evaluate: {MODELS}")

DDI Images: /content/drive/MyDrive/thesis/data/ddidiversedermatologyimages
Models directory: /content/drive/MyDrive/thesis/results
Results will be saved to: /content/drive/MyDrive/thesis/results/ddi_evaluation

Seeds to evaluate: [42, 123]
Models to evaluate: ['baseline', 'mixup', 'reweighted', 'focalloss', 'proposed']


## 3. Helper Functions

In [5]:
def load_model(model_path):
    """Load a trained model from checkpoint."""
    model = SkinLesionClassifier(num_classes=2, pretrained=False)
    state_dict = torch.load(model_path, map_location=device)
    model.load_state_dict(state_dict)
    model = model.to(device)
    model.eval()
    return model


def evaluate_model(model, dataloader, device):
    """Run inference and collect predictions."""
    model.eval()

    all_preds = []
    all_labels = []
    all_groups = []
    all_probs = []

    with torch.no_grad():
        for images, labels, groups in tqdm(dataloader, desc='Evaluating'):
            images = images.to(device)

            outputs = model(images)
            probs = F.softmax(outputs, dim=1)
            _, predicted = outputs.max(1)

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.numpy())
            all_groups.extend(groups.numpy())
            all_probs.extend(probs.cpu().numpy())

    return {
        'preds': np.array(all_preds),
        'labels': np.array(all_labels),
        'groups': np.array(all_groups),
        'probs': np.array(all_probs)
    }


def compute_mean_std(values):
    """Compute mean and std from list of values."""
    if len(values) == 0:
        return None, None
    return np.mean(values), np.std(values)


def format_mean_std(mean, std):
    """Format mean ± std as string."""
    if mean is None:
        return 'N/A'
    return f"{mean:.4f} ± {std:.4f}"

## 4. Load DDI Dataset

In [6]:
# Create DDI dataset with evaluation transforms
eval_transforms = get_ddi_eval_transforms()

ddi_dataset = DDIDataset(
    metadata_path=DDI_METADATA_PATH,
    images_dir=DDI_IMAGES_DIR,
    transform=eval_transforms
)

# Create dataloader
ddi_loader = DataLoader(
    ddi_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# Print dataset statistics
stats = ddi_dataset.get_statistics()
print(f"\nDataset Statistics:")
print(f"  Total samples: {stats['total']}")
print(f"  Per group: {stats['per_group']}")
print(f"  Per class: {stats['per_class']}")


DDI Dataset loaded: 656 images

Skin tone distribution:
skin_tone_group
Medium    241
Light     208
Dark      207
Name: count, dtype: int64

Class distribution:
label
Benign       485
Malignant    171
Name: count, dtype: int64

Dataset Statistics:
  Total samples: 656
  Per group: {'Medium': 241, 'Light': 208, 'Dark': 207}
  Per class: {0: 485, 1: 171}


## 5. Evaluate All Models Across All Seeds

In [7]:
# Store all results: {model: {seed: metrics}}
all_ddi_results = {model: {} for model in MODELS}

for model_name in MODELS:
    for seed in SEEDS:
        run_name = f"{model_name}_seed{seed}"
        model_path = RESULTS_DIR / run_name / 'model.pt'

        # Check if model exists
        if not model_path.exists():
            print(f"\nWarning: Model not found: {model_path}")
            print(f"  Skipping {run_name}...")
            continue

        print(f"\n{'=' * 60}")
        print(f"Evaluating: {run_name}")
        print(f"{'=' * 60}")

        # Load and evaluate
        model = load_model(model_path)
        eval_results = evaluate_model(model, ddi_loader, device)

        # Compute metrics
        overall = compute_metrics(
            eval_results['labels'],
            eval_results['preds'],
            eval_results['probs']
        )

        fairness = compute_fairness_metrics(
            eval_results['labels'],
            eval_results['preds'],
            eval_results['groups']
        )

        # Store results
        all_ddi_results[model_name][seed] = {
            'overall': overall,
            'fairness': fairness
        }

        print(f"  Accuracy: {overall['accuracy']:.4f}")
        print(f"  Balanced Accuracy: {overall['balanced_accuracy']:.4f}")
        print(f"  EO Gap: {fairness['equal_opportunity_gap']:.4f}")

        # Clean up GPU memory
        del model
        torch.cuda.empty_cache()

print("\n" + "=" * 60)
print("All models evaluated!")
print("=" * 60)


Evaluating: baseline_seed42


Evaluating: 100%|██████████| 21/21 [02:29<00:00,  7.12s/it]


  Accuracy: 0.6799
  Balanced Accuracy: 0.6472
  EO Gap: 0.3393

Evaluating: baseline_seed123


Evaluating: 100%|██████████| 21/21 [00:07<00:00,  2.82it/s]


  Accuracy: 0.6204
  Balanced Accuracy: 0.6221
  EO Gap: 0.3588

Evaluating: mixup_seed42


...

  Accuracy: 0.5762
  Balanced Accuracy: 0.5960
  EO Gap: 0.4409

Evaluating: proposed_seed42


Evaluating: 100%|██████████| 21/21 [00:07<00:00,  2.83it/s]


  Accuracy: 0.6113
  Balanced Accuracy: 0.6141
  EO Gap: 0.1965

Evaluating: proposed_seed123


Evaluating: 100%|██████████| 21/21 [00:07<00:00,  2.82it/s]

  Accuracy: 0.5793
  Balanced Accuracy: 0.5981
  EO Gap: 0.4001

All models evaluated!


## 6. Aggregate Results (Mean ± Std)

In [8]:
def aggregate_ddi_metrics(model_results):
    """Aggregate metrics across seeds for a single model."""
    seeds_data = list(model_results.values())
    if len(seeds_data) == 0:
        return None

    result = {
        'overall': {},
        'fairness': {},
        'n_seeds': len(seeds_data)
    }

    # Overall metrics
    for key in ['accuracy', 'balanced_accuracy', 'f1_macro', 'f1_weighted', 'roc_auc']:
        values = [s['overall'].get(key) for s in seeds_data if s['overall'].get(key) is not None]
        if values:
            result['overall'][key] = {
                'mean': np.mean(values),
                'std': np.std(values),
                'values': values
            }

    # Fairness metrics
    for key in ['equal_opportunity_gap', 'equalized_odds_gap', 'f1_gap']:
        values = [s['fairness'].get(key) for s in seeds_data if s['fairness'].get(key) is not None]
        if values:
            result['fairness'][key] = {
                'mean': np.mean(values),
                'std': np.std(values),
                'values': values
            }

    # Per-group TPR
    result['fairness']['tpr_per_group'] = {}
    for group in ['Light', 'Medium', 'Dark']:
        values = [s['fairness']['tpr_per_group'].get(group) for s in seeds_data
                  if s['fairness']['tpr_per_group'].get(group) is not None]
        if values:
            result['fairness']['tpr_per_group'][group] = {
                'mean': np.mean(values),
                'std': np.std(values),
                'values': values
            }

    return result

# Aggregate all models
aggregated_ddi = {}
for model in MODELS:
    if all_ddi_results[model]:
        aggregated_ddi[model] = aggregate_ddi_metrics(all_ddi_results[model])
        print(f"Aggregated {model}: {aggregated_ddi[model]['n_seeds']} seeds")
    else:
        print(f"No results for {model}")

Aggregated baseline: 2 seeds
Aggregated mixup: 2 seeds
Aggregated reweighted: 2 seeds
Aggregated focalloss: 2 seeds
Aggregated proposed: 2 seeds


## 7. Display Aggregated Results

In [9]:
print("=" * 80)
print("DDI EVALUATION RESULTS (Mean ± Std over 2 seeds)")
print("=" * 80)

# Build comparison table
rows = []
for model in MODELS:
    if model in aggregated_ddi and aggregated_ddi[model]:
        m = aggregated_ddi[model]
        row = {
            'Model': model.capitalize(),
            'Balanced Acc': format_mean_std(
                m['overall']['balanced_accuracy']['mean'],
                m['overall']['balanced_accuracy']['std']
            ),
            'ROC-AUC': format_mean_std(
                m['overall']['roc_auc']['mean'],
                m['overall']['roc_auc']['std']
            ) if 'roc_auc' in m['overall'] else 'N/A',
            'EO Gap': format_mean_std(
                m['fairness']['equal_opportunity_gap']['mean'],
                m['fairness']['equal_opportunity_gap']['std']
            ),
            'TPR Dark': format_mean_std(
                m['fairness']['tpr_per_group']['Dark']['mean'],
                m['fairness']['tpr_per_group']['Dark']['std']
            ),
        }
        rows.append(row)

ddi_comparison_df = pd.DataFrame(rows)
print(ddi_comparison_df.to_string(index=False))

DDI EVALUATION RESULTS (Mean ± Std over 2 seeds)
     Model    Balanced Acc         ROC-AUC          EO Gap        TPR Dark
  Baseline 0.6347 ± 0.0125 0.6641 ± 0.0111 0.3491 ± 0.0098 0.3958 ± 0.0208
     Mixup 0.6111 ± 0.0001 0.6424 ± 0.0044 0.2803 ± 0.1193 0.4792 ± 0.0625
Reweighted 0.6397 ± 0.0117 0.6666 ± 0.0109 0.3491 ± 0.0098 0.3958 ± 0.0208
 Focalloss 0.6155 ± 0.0195 0.6466 ± 0.0150 0.4105 ± 0.0304 0.3854 ± 0.0104
  Proposed 0.6061 ± 0.0080 0.6381 ± 0.0120 0.2983 ± 0.1018 0.4375 ± 0.0417


## 8. Save All Results

In [10]:
# Helper for JSON serialization
def convert_to_serializable(obj):
    if isinstance(obj, np.floating):
        return float(obj)
    elif isinstance(obj, np.integer):
        return int(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, dict):
        return {k: convert_to_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_to_serializable(i) for i in obj]
    return obj

# 1. Save per-seed results (detailed)
per_seed_rows = []
for model in MODELS:
    for seed in SEEDS:
        if seed in all_ddi_results[model]:
            r = all_ddi_results[model][seed]
            per_seed_rows.append({
                'model': model,
                'seed': seed,
                'accuracy': r['overall']['accuracy'],
                'balanced_accuracy': r['overall']['balanced_accuracy'],
                'roc_auc': r['overall'].get('roc_auc'),
                'f1_macro': r['overall']['f1_macro'],
                'equal_opportunity_gap': r['fairness']['equal_opportunity_gap'],
                'equalized_odds_gap': r['fairness']['equalized_odds_gap'],
                'f1_gap': r['fairness']['f1_gap'],
                'tpr_light': r['fairness']['tpr_per_group'].get('Light'),
                'tpr_medium': r['fairness']['tpr_per_group'].get('Medium'),
                'tpr_dark': r['fairness']['tpr_per_group'].get('Dark'),
            })

per_seed_df = pd.DataFrame(per_seed_rows)
per_seed_path = DDI_RESULTS_DIR / 'ddi_evaluation_per_seed.csv'
per_seed_df.to_csv(per_seed_path, index=False)
print(f"Per-seed results saved to: {per_seed_path}")

# 2. Save aggregated results (mean ± std)
agg_rows = []
for model in MODELS:
    if model in aggregated_ddi and aggregated_ddi[model]:
        m = aggregated_ddi[model]
        agg_rows.append({
            'model': model,
            'n_seeds': m['n_seeds'],
            'balanced_accuracy_mean': m['overall']['balanced_accuracy']['mean'],
            'balanced_accuracy_std': m['overall']['balanced_accuracy']['std'],
            'roc_auc_mean': m['overall'].get('roc_auc', {}).get('mean'),
            'roc_auc_std': m['overall'].get('roc_auc', {}).get('std'),
            'eo_gap_mean': m['fairness']['equal_opportunity_gap']['mean'],
            'eo_gap_std': m['fairness']['equal_opportunity_gap']['std'],
            'tpr_dark_mean': m['fairness']['tpr_per_group']['Dark']['mean'],
            'tpr_dark_std': m['fairness']['tpr_per_group']['Dark']['std'],
        })

agg_df = pd.DataFrame(agg_rows)
agg_path = DDI_RESULTS_DIR / 'ddi_evaluation_aggregated.csv'
agg_df.to_csv(agg_path, index=False)
print(f"Aggregated results saved to: {agg_path}")

# 3. Save full JSON with all details
json_path = DDI_RESULTS_DIR / 'ddi_evaluation_multiseed.json'
with open(json_path, 'w') as f:
    json.dump(convert_to_serializable(aggregated_ddi), f, indent=2)
print(f"Full JSON saved to: {json_path}")

# 4. Save display-ready comparison table
display_path = DDI_RESULTS_DIR / 'ddi_comparison_display.csv'
ddi_comparison_df.to_csv(display_path, index=False)
print(f"Display table saved to: {display_path}")

Per-seed results saved to: /content/drive/MyDrive/thesis/results/ddi_evaluation/ddi_evaluation_per_seed.csv
Aggregated results saved to: /content/drive/MyDrive/thesis/results/ddi_evaluation/ddi_evaluation_aggregated.csv
Full JSON saved to: /content/drive/MyDrive/thesis/results/ddi_evaluation/ddi_evaluation_multiseed.json
Display table saved to: /content/drive/MyDrive/thesis/results/ddi_evaluation/ddi_comparison_display.csv


## 9. Load Fitzpatrick17k Results for Comparison

In [11]:
# Load Fitzpatrick17k aggregated results for comparison
fitz_agg_path = RESULTS_DIR / 'aggregated_results_2seeds.json'

if fitz_agg_path.exists():
    with open(fitz_agg_path, 'r') as f:
        fitz_aggregated = json.load(f)
    print("Loaded Fitzpatrick17k aggregated results")

    # Build comparison table: Fitz vs DDI
    comparison_rows = []
    for model in MODELS:
        if model in fitz_aggregated and model in aggregated_ddi:
            fitz = fitz_aggregated[model]
            ddi = aggregated_ddi[model]

            fitz_acc = fitz['overall']['balanced_accuracy']['mean']
            ddi_acc = ddi['overall']['balanced_accuracy']['mean']

            comparison_rows.append({
                'Model': model.capitalize(),
                'Fitz Bal Acc': f"{fitz_acc:.4f}",
                'DDI Bal Acc': f"{ddi_acc:.4f}",
                'Accuracy Drop': f"{(fitz_acc - ddi_acc) * 100:.1f}%",
                'Fitz EO Gap': f"{fitz['fairness']['equal_opportunity_gap']['mean']:.4f}",
                'DDI EO Gap': f"{ddi['fairness']['equal_opportunity_gap']['mean']:.4f}",
            })

    comparison_df = pd.DataFrame(comparison_rows)
    print("\n" + "=" * 80)
    print("FITZPATRICK17K vs DDI COMPARISON (Mean over 2 seeds)")
    print("=" * 80)
    print(comparison_df.to_string(index=False))

    # Save comparison
    comparison_path = DDI_RESULTS_DIR / 'fitz_vs_ddi_comparison.csv'
    comparison_df.to_csv(comparison_path, index=False)
    print(f"\nComparison saved to: {comparison_path}")
else:
    print(f"Fitzpatrick17k aggregated results not found at {fitz_agg_path}")
    print("Run fairderm_training_seed123.ipynb first to generate aggregated results.")

Loaded Fitzpatrick17k aggregated results

FITZPATRICK17K vs DDI COMPARISON (Mean over 2 seeds)
     Model Fitz Bal Acc DDI Bal Acc Accuracy Drop Fitz EO Gap DDI EO Gap
  Baseline       0.8272      0.6347         19.2%      0.0918     0.3491
     Mixup       0.8561      0.6111         24.5%      0.1005     0.2803
Reweighted       0.8261      0.6397         18.6%      0.0835     0.3491
 Focalloss       0.8266      0.6155         21.1%      0.0846     0.4105
  Proposed       0.8627      0.6061         25.7%      0.0714     0.2983

Comparison saved to: /content/drive/MyDrive/thesis/results/ddi_evaluation/fitz_vs_ddi_comparison.csv


## 10. Summary

In [12]:
print("=" * 60)
print("DDI MULTI-SEED EVALUATION COMPLETE")
print("=" * 60)
print(f"\nSeeds evaluated: {SEEDS}")
print(f"Models evaluated: {MODELS}")
print(f"\nOutput files:")
print(f"  - {per_seed_path.name}: Per-seed detailed results")
print(f"  - {agg_path.name}: Aggregated mean ± std")
print(f"  - {json_path.name}: Full JSON with all metrics")
print(f"  - {display_path.name}: Display-ready table")
print(f"\nFor visualizations, run: fairderm_results_analysis_multiseed.ipynb")

DDI MULTI-SEED EVALUATION COMPLETE

Seeds evaluated: [42, 123]
Models evaluated: ['baseline', 'mixup', 'reweighted', 'focalloss', 'proposed']

Output files:
  - ddi_evaluation_per_seed.csv: Per-seed detailed results
  - ddi_evaluation_aggregated.csv: Aggregated mean ± std
  - ddi_evaluation_multiseed.json: Full JSON with all metrics
  - ddi_comparison_display.csv: Display-ready table

For visualizations, run: fairderm_results_analysis_multiseed.ipynb
